# Paper-to-Code 4 – DDPM (2020)
**Paper:** Ho, Jain, Abbeel – *Denoising Diffusion Probabilistic Models* – [arXiv:2006.11239](https://arxiv.org/abs/2006.11239) · notes: [[Generative Models]] · plan: [[Paper-to-Code Months]]

### The claim we reproduce
> Gradually add Gaussian noise to data over $T=1000$ steps; train a network to **predict the noise** with a plain MSE loss (the "simplified objective", Eq. 14); then generate by running the learned denoising chain backwards from pure noise (Algorithm 2). This simple recipe produces high-quality samples.

### Our scaled-down setup
Same algorithm, same $T=1000$ and linear $\beta$ schedule $10^{-4} \to 0.02$ – but on **2-D data** (two moons) with a small MLP instead of images with a U-Net, so training takes ≈ 10 s on a CPU and you can *see* every step. The paper's CIFAR-10 model reached FID 3.17.

**How this works:** cells marked **TODO** are yours (each one is a small function). Run the ✅ check under it.
Experiment cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*.
Everything runs on a laptop CPU – the paper's full-scale setup is listed under *Scale it up*.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, time, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.datasets import make_moons
torch.manual_seed(0); torch.set_num_threads(4)
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the experiment cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing

Xnp, _ = make_moons(20000, noise=0.05, random_state=0)
X = torch.tensor((Xnp - Xnp.mean(0)) / Xnp.std(0), dtype=torch.float)     # standardised, like images scaled to [-1, 1]
plt.figure(figsize=(4, 3)); plt.scatter(*X[:2000].T, s=2); plt.title("data"); plt.axis("equal"); plt.show()

## 1. The noise schedule
$\beta_t$ rises linearly from $\beta_1 = 10^{-4}$ to $\beta_T = 0.02$. Define $\alpha_t = 1-\beta_t$ and $\bar\alpha_t = \prod_{s\le t}\alpha_s$. Return `betas, alphas, alpha_bar`, each of length $T$ (index 0 is the paper's $t=1$).

In [ ]:
def make_schedule(T=1000, beta_1=1e-4, beta_T=0.02):
    # TODO 1: linear betas, alphas = 1 - betas, alpha_bar = cumulative product
    raise NotImplementedError  # TODO 1

In [ ]:
def _t_sched():
    b, a, ab = make_schedule()
    return len(ab) == 1000 and abs(ab[0] - (1 - 1e-4)) < 1e-6 and bool((ab[1:] < ab[:-1]).all()) and ab[-1] < 1e-4
check("schedule", _t_sched, "alpha_bar must start at 0.9999, strictly decrease and end below 1e-4 (≈ 4e-5).")
T = 1000
try: betas, alphas, alpha_bar = make_schedule(T)
except NotImplementedError: pass

## 2. Jumping straight to step $t$ (paper Eq. 4)
The forward process has a closed form – no need to add noise 1000 times:
$$x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon, \qquad \varepsilon\sim\mathcal N(0, I)$$
`x0` and `eps` are `(B, 2)`, `t` is a `(B,)` integer tensor.

In [ ]:
def q_sample(x0, t, eps):
    # TODO 2: the closed-form forward process above (broadcast alpha_bar[t] over the last dimension)
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_q():
    x0, eps = torch.randn(4, 2), torch.randn(4, 2)
    t = torch.tensor([0, 10, 500, 999])
    want = alpha_bar[t, None].sqrt() * x0 + (1 - alpha_bar[t, None]).sqrt() * eps
    return torch.allclose(q_sample(x0, t, eps), want) and torch.allclose(q_sample(x0, torch.full((4,), 999), eps), eps, atol=0.05)
check("forward process", _t_q, "Expected sqrt(alpha_bar[t])·x0 + sqrt(1 − alpha_bar[t])·eps per row.")
if ready("schedule", "forward process"):
    fig, ax = plt.subplots(1, 5, figsize=(14, 2.6))
    for a, t in zip(ax, [0, 50, 200, 500, 999]):
        a.scatter(*q_sample(X[:2000], torch.full((2000,), t), torch.randn(2000, 2)).T, s=1); a.set_title(f"t = {t+1}"); a.axis("off")
    plt.suptitle("forward process: data → noise"); plt.show()

## 3. The noise-prediction network and the simplified loss (Algorithm 1, Eq. 14)
$\varepsilon_\theta(x_t, t)$ is an MLP that also sees a sinusoidal embedding of $t$ (the same trick as Transformer positional encodings). One training step: pick random $x_0$, $t \sim \text{Uniform}\{0..T-1\}$, $\varepsilon\sim\mathcal N(0,I)$ and minimise
$$\lVert \varepsilon - \varepsilon_\theta(\sqrt{\bar\alpha_t}x_0 + \sqrt{1-\bar\alpha_t}\varepsilon,\; t)\rVert^2$$

In [ ]:
def time_embedding(t, dim=32):
    freqs = torch.exp(-math.log(10000) * torch.arange(dim // 2) / (dim // 2))
    angles = t[:, None].float() * freqs[None]
    return torch.cat([angles.sin(), angles.cos()], 1)

class EpsModel(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2 + 32, hidden), nn.SiLU(), nn.Linear(hidden, hidden), nn.SiLU(),
                                 nn.Linear(hidden, hidden), nn.SiLU(), nn.Linear(hidden, 2))
    def forward(self, x, t):
        return self.net(torch.cat([x, time_embedding(t)], 1))

def ddpm_loss(model, x0):
    # TODO 3: sample t and eps, build x_t with q_sample, return the MSE between model(x_t, t) and eps
    raise NotImplementedError  # TODO 3

In [ ]:
def _t_loss():
    zero_model = lambda x, t: torch.zeros_like(x)          # predicting 0 → loss = E[eps²] = 1
    torch.manual_seed(1); l = ddpm_loss(zero_model, X[:20000])
    return l.dim() == 0 and abs(l.item() - 1) < 0.03
check("loss", _t_loss, "A model that always predicts 0 should get loss ≈ 1 (the variance of eps) – are you comparing to eps, not x0?")

## 4. Sampling (Algorithm 2)
Start from $x_T \sim \mathcal N(0, I)$ and for $t = T..1$:
$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\Big(x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\varepsilon_\theta(x_t, t)\Big) + \sigma_t z, \qquad \sigma_t^2 = \beta_t,\; z\sim\mathcal N(0,I)$$
and **no noise on the last step** ($t=1$, index 0). Implement one step for a batch `x` at integer step `t`.

In [ ]:
@torch.no_grad()
def p_sample_step(model, x, t):
    # TODO 4: one reverse step x_t → x_{t-1} (Algorithm 2, line 4)
    raise NotImplementedError  # TODO 4

@torch.no_grad()
def sample(model, n, snapshots=()):
    x, snaps = torch.randn(n, 2), {}
    for t in reversed(range(T)):
        x = p_sample_step(model, x, t)
        if t in snapshots: snaps[t] = x.clone()
    return x, snaps

In [ ]:
def _t_step():
    zero_model = lambda x, t: torch.zeros_like(x); x = torch.randn(5, 2)
    last = p_sample_step(zero_model, x, 0)                               # t = 0: deterministic, x / sqrt(alpha_0)
    torch.manual_seed(0); a = p_sample_step(zero_model, x, 500)
    torch.manual_seed(0); b = x / alphas[500].sqrt() + betas[500].sqrt() * torch.randn_like(x)
    return torch.allclose(last, x / alphas[0].sqrt()) and torch.allclose(a, b)
check("reverse step", _t_step, "With eps_hat = 0 the step is x/sqrt(alpha_t) + sqrt(beta_t)·z, and no z at t = 0.")

## 5. Train and generate

In [ ]:
STEPS = 4000
if ready("schedule", "forward process", "loss", "reverse step"):
    torch.manual_seed(0)
    model = EpsModel(); opt = torch.optim.Adam(model.parameters(), lr=2e-3)
    t0, curve = time.time(), []
    for step in range(STEPS):
        loss = ddpm_loss(model, X[torch.randint(0, len(X), (512,))])
        opt.zero_grad(); loss.backward(); opt.step(); curve.append(loss.item())
    print(f"trained {STEPS} steps in {time.time()-t0:.0f}s, final loss {np.mean(curve[-200:]):.3f}")

    samples, snaps = sample(model, 2000, snapshots=(999, 500, 250, 100, 0))
    fig, ax = plt.subplots(1, 5, figsize=(14, 2.6))
    for a, t in zip(ax, (999, 500, 250, 100, 0)):
        a.scatter(*snaps[t].T, s=1, c="C1"); a.set_title(f"after step t = {t+1}"); a.axis("off")
    plt.suptitle("reverse process: noise → data"); plt.show()

**How do we score samples?** No FID for 2-D points – instead the mean distance from each sample to its nearest real point (*precision*: are samples on the moons?) and from each real point to its nearest sample (*coverage*: are both moons filled?). Pure noise scores ≈ 0.25.

In [ ]:
def nn_dist(A, B):
    return torch.cdist(A, B).min(1).values.mean().item()

if ready("schedule", "forward process", "loss", "reverse step"):
    real = X[2000:7000]
    precision, coverage, noise_score = nn_dist(samples, real), nn_dist(X[:2000], samples), nn_dist(torch.randn(2000, 2), real)
    upper = (samples[:, 1] > samples[:, 1].median()).float().mean()   # sanity: not collapsed to a single point/moon
    print(f"sample→data {precision:.3f}   data→sample {coverage:.3f}   (pure noise→data {noise_score:.3f})")
    check("samples look like the data", lambda: precision < 0.06 and coverage < 0.06,
          "Samples should land on the moons (< 0.06) and cover them – check the sign/scale in the reverse step.")

<details><summary>▶ Solution</summary>

```python
def make_schedule(T=1000, beta_1=1e-4, beta_T=0.02):
    betas = torch.linspace(beta_1, beta_T, T)
    alphas = 1 - betas
    return betas, alphas, torch.cumprod(alphas, 0)
```

```python
def q_sample(x0, t, eps):
    ab = alpha_bar[t].unsqueeze(-1)
    return ab.sqrt() * x0 + (1 - ab).sqrt() * eps
```

```python
def ddpm_loss(model, x0):
    t = torch.randint(0, T, (len(x0),))
    eps = torch.randn_like(x0)
    return F.mse_loss(model(q_sample(x0, t, eps), t), eps)
```

```python
@torch.no_grad()
def p_sample_step(model, x, t):
    tt = torch.full((len(x),), t)
    eps_hat = model(x, tt)
    mean = (x - betas[t] / (1 - alpha_bar[t]).sqrt() * eps_hat) / alphas[t].sqrt()
    if t == 0:
        return mean
    return mean + betas[t].sqrt() * torch.randn_like(x)
```
</details>

## Scale it up (stretch goals)
- **MNIST** with a small U-Net (2 down/up levels, time embedding added in every block); 20–30 epochs on a GPU give recognisable digits.
- Swap the linear schedule for the **cosine** schedule of Improved DDPM ([arXiv:2102.09672](https://arxiv.org/abs/2102.09672)) and compare at $T=100$.
- Predict $x_0$ instead of $\varepsilon$ – the paper (Table 2) found $\varepsilon$-prediction with the simple loss works best. Do you see the same?
- Fewer sampling steps: skip steps DDIM-style and watch quality drop (or not).

---
## 📝 Write-up (fill in – half a page is enough)
Copy this into your results log in [[Paper-to-Code Months]] when done.

- **Paper and its claim in one sentence:**
- **What I implemented / what I scaled down (and why):**
- **Result:**

  | setting | metric | paper | me |
  |---|---|---|---|
  |  |  |  |  |

- **Did the claim hold at small scale? Why / why not?**
- **One detail the paper glossed over that I had to figure out:**
- **What I would try next with a GPU:**
